# 02 - Baseline Models (TF-IDF + Word2Vec)
**Author:** Sanjeev Kumar | IIT Bombay

Benchmark classical NLP before BERT.

In [ ]:
import matplotlib
matplotlib.use('Agg')
import pandas as pd, numpy as np, re, os, sys, warnings
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')
os.makedirs('../reports', exist_ok=True)
sys.path.insert(0, '..')
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score, classification_report
from xgboost import XGBClassifier
print("Imports done!")

In [ ]:
PRODUCT_MAP = {
    'Credit reporting or other personal consumer reports': 'Credit Reporting',
    'Credit reporting, credit repair services, or other personal consumer reports': 'Credit Reporting',
    'Credit reporting': 'Credit Reporting',
    'Debt collection': 'Debt Collection',
    'Mortgage': 'Mortgage',
    'Checking or savings account': 'Bank Account',
    'Bank account or service': 'Bank Account',
    'Credit card': 'Credit Card',
    'Credit card or prepaid card': 'Credit Card',
    'Prepaid card': 'Credit Card',
    'Student loan': 'Loans',
    'Vehicle loan or lease': 'Loans',
    'Consumer Loan': 'Loans',
    'Payday loan, title loan, personal loan, or advance loan': 'Loans',
    'Payday loan, title loan, or personal loan': 'Loans',
    'Payday loan': 'Loans',
    'Money transfer, virtual currency, or money service': 'Money Transfer',
    'Money transfers': 'Money Transfer',
    'Debt or credit management': 'Debt Collection',
    'Other financial service': 'Other',
}

def clean_text(text):
    if pd.isna(text): return ""
    text = str(text).lower()
    text = re.sub(r'x{2,}', 'XXXX', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

os.makedirs('../data', exist_ok=True)
local_path = '../data/complaints_200k.csv'
if os.path.exists(local_path):
    print("Loading from local cache...")
    df = pd.read_csv(local_path, low_memory=False)
else:
    import requests
    url = "https://files.consumerfinance.gov/ccdb/complaints.csv.zip"
    zip_path = '../data/complaints.csv.zip'
    print("Downloading CFPB data (streaming)...")
    with requests.get(url, stream=True, timeout=600) as r:
        r.raise_for_status()
        with open(zip_path, 'wb') as f:
            for chunk in r.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)
    df = pd.read_csv(zip_path, compression='zip', low_memory=False, nrows=200000)
    df.to_csv(local_path, index=False)

df_text = df[df['Consumer complaint narrative'].notna()].copy()
df_text['product_clean'] = df_text['Product'].map(PRODUCT_MAP).fillna('Other')
df_text['narrative_clean'] = df_text['Consumer complaint narrative'].apply(clean_text)
df_text['Date received'] = pd.to_datetime(df_text['Date received'])
df_text['is_priority_review'] = (df_text['Timely response?'] == 'No').astype(int)
print(f"Rows with narrative: {len(df_text):,}")
print(f"Priority review rate: {df_text['is_priority_review'].mean():.2%}")

In [ ]:
# Chronological split
df_sorted = df_text.sort_values('Date received').reset_index(drop=True)
n = len(df_sorted)
train_df = df_sorted.iloc[:int(n*0.70)].copy()
val_df   = df_sorted.iloc[int(n*0.70):int(n*0.85)].copy()
test_df  = df_sorted.iloc[int(n*0.85):].copy()

X_train = train_df['narrative_clean'].values
X_val   = val_df['narrative_clean'].values
X_test  = test_df['narrative_clean'].values
y_train_prod = train_df['product_clean'].values
y_val_prod   = val_df['product_clean'].values
y_test_prod  = test_df['product_clean'].values
y_train_risk = train_df['is_priority_review'].values
y_val_risk   = val_df['is_priority_review'].values
y_test_risk  = test_df['is_priority_review'].values

print(f"Train: {len(train_df):,} | {train_df['Date received'].min().date()} - {train_df['Date received'].max().date()}")
print(f"Val:   {len(val_df):,}   | {val_df['Date received'].min().date()} - {val_df['Date received'].max().date()}")
print(f"Test:  {len(test_df):,}  | {test_df['Date received'].min().date()} - {test_df['Date received'].max().date()}")

In [ ]:
# BASELINE 1: TF-IDF + Logistic Regression
print("="*60)
print("BASELINE 1: TF-IDF + Logistic Regression")
tfidf_lr = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=50000, ngram_range=(1,2), min_df=2, sublinear_tf=True)),
    ('clf', LogisticRegression(max_iter=1000, C=1.0, class_weight='balanced', random_state=42))
])
tfidf_lr.fit(X_train, y_train_prod)
pred_val  = tfidf_lr.predict(X_val)
pred_test = tfidf_lr.predict(X_test)
val_f1_lr  = f1_score(y_val_prod,  pred_val,  average='macro')
test_f1_lr = f1_score(y_test_prod, pred_test, average='macro')
print(f"Val Macro-F1:  {val_f1_lr:.4f}")
print(f"Test Macro-F1: {test_f1_lr:.4f}")
print(classification_report(y_test_prod, pred_test))

In [ ]:
# BASELINE 2: TF-IDF + LinearSVC
print("=" * 60)
print("BASELINE 2: TF-IDF + Linear SVM")
svm_pipe = Pipeline([
    ('tfidf', TfidfVectorizer(
        max_features=50000,
        ngram_range=(1, 2),
        min_df=2,
        sublinear_tf=True
    )),
    ('clf', LinearSVC(
        max_iter=2000,
        class_weight='balanced',
        random_state=42,
        dual=False
    ))
])
svm_pipe.fit(X_train, y_train_prod)
val_f1_svm  = f1_score(y_val_prod,  svm_pipe.predict(X_val),  average='macro')
test_f1_svm = f1_score(y_test_prod, svm_pipe.predict(X_test), average='macro')
print(f"Val Macro-F1:  {val_f1_svm:.4f}")
print(f"Test Macro-F1: {test_f1_svm:.4f}")

In [ ]:
# BASELINE 3: Word2Vec + XGBoost
print("="*60)
print("BASELINE 3: Word2Vec + XGBoost")
from gensim.models import Word2Vec
X_train_tok = [t.split() for t in X_train]
X_val_tok   = [t.split() for t in X_val]
X_test_tok  = [t.split() for t in X_test]
w2v = Word2Vec(sentences=X_train_tok, vector_size=300, window=5, min_count=2, workers=4, epochs=10, seed=42)
print(f"Vocabulary: {len(w2v.wv):,} words")
def text_to_vec(tokens, model, dim=300):
    vecs = [model.wv[w] for w in tokens if w in model.wv]
    return np.mean(vecs, axis=0) if vecs else np.zeros(dim)
X_train_w2v = np.array([text_to_vec(t, w2v) for t in X_train_tok])
X_val_w2v   = np.array([text_to_vec(t, w2v) for t in X_val_tok])
X_test_w2v  = np.array([text_to_vec(t, w2v) for t in X_test_tok])
le = LabelEncoder()
y_tr_enc = le.fit_transform(y_train_prod)
xgb = XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.1, subsample=0.8, colsample_bytree=0.8, eval_metric='mlogloss', random_state=42, verbosity=0)
xgb.fit(X_train_w2v, y_tr_enc, eval_set=[(X_val_w2v, le.transform(y_val_prod))], verbose=False)
pred_t_w2v = le.inverse_transform(xgb.predict(X_test_w2v))
val_f1_w2v  = f1_score(y_val_prod, le.inverse_transform(xgb.predict(X_val_w2v)), average='macro')
test_f1_w2v = f1_score(y_test_prod, pred_t_w2v, average='macro')
print(f"Val Macro-F1:  {val_f1_w2v:.4f}")
print(f"Test Macro-F1: {test_f1_w2v:.4f}")

In [ ]:
import joblib
os.makedirs('../models', exist_ok=True)
joblib.dump(tfidf_lr,                        '../models/product_classifier_champion.pkl')
joblib.dump(svm_pipe,                        '../models/product_classifier_svm.pkl')
joblib.dump(tfidf_lr.named_steps['tfidf'],   '../models/tfidf_vectorizer.pkl')
joblib.dump(le,                              '../models/label_encoder.pkl')

print("=" * 65)
print("BASELINE COMPARISON TABLE")
print("=" * 65)
print(f"{'Model':<35} {'Val F1':>8} {'Test F1':>8}")
print("-" * 55)
print(f"{'TF-IDF + LR (CHAMPION)':<35} {val_f1_lr:>8.4f} {test_f1_lr:>8.4f}")
print(f"{'TF-IDF + SVM':<35} {val_f1_svm:>8.4f} {test_f1_svm:>8.4f}")
print(f"{'Word2Vec + XGBoost':<35} {val_f1_w2v:>8.4f} {test_f1_w2v:>8.4f}")
print("\nModels saved!")
print("Key insight: Word2Vec WORSE than TF-IDF!")
print("Reason: keyword-heavy task + averaging loses signal!")

In [ ]:
# Error Analysis - Misclassified Examples
from sklearn.metrics import confusion_matrix
import numpy as np

# Get predictions
pred_test = tfidf_lr.predict(X_test)

# Confusion matrix
cm = confusion_matrix(y_test_prod, pred_test,
     labels=['Bank Account','Credit Card',
     'Credit Reporting','Debt Collection',
     'Loans','Money Transfer','Mortgage'])

print("CONFUSION MATRIX:")
print("="*60)
classes = ['Bank Acct','Credit Card',
           'Credit Rep','Debt Coll',
           'Loans','Money Trans','Mortgage']
print(f"{'':12}", end='')
for c in classes:
    print(f"{c:12}", end='')
print()
for i, row in enumerate(cm):
    print(f"{classes[i]:12}", end='')
    for val in row:
        print(f"{val:<12}", end='')
    print()

# Find misclassified examples
print("\n\nTOP MISCLASSIFIED EXAMPLES:")
print("="*60)
misclassified = []
for i, (true, pred, text) in enumerate(
    zip(y_test_prod, pred_test, X_test)):
    if true != pred:
        misclassified.append({
            'true': true,
            'pred': pred,
            'text': text[:200]
        })

# Show top confused pairs
from collections import Counter
confused_pairs = Counter([
    (m['true'], m['pred'])
    for m in misclassified
])
print("\nTop confused class pairs:")
for (true, pred), count in confused_pairs.most_common(5):
    print(f"  True={true} → Pred={pred}: {count} cases")

print("\nSample misclassified complaints:")
shown = {}
for m in misclassified:
    pair = (m['true'], m['pred'])
    if pair not in shown and len(shown) < 5:
        shown[pair] = True
        print(f"\nTrue: {m['true']}")
        print(f"Pred: {m['pred']}")
        print(f"Text: {m['text']}...")

# Per-class F1 table
from sklearn.metrics import (
    precision_score, recall_score, f1_score)
print("\n\nPER-CLASS F1 TABLE:")
print("="*60)
print(f"{'Product':<20} {'Precision':>10} {'Recall':>8} {'F1':>8} {'Support':>10}")
print("-"*60)
classes_full = sorted(set(y_test_prod))
for cls in classes_full:
    mask = np.array(y_test_prod) == cls
    sup  = mask.sum()
    p = precision_score(y_test_prod, pred_test,
        labels=[cls], average='macro',
        zero_division=0)
    r = recall_score(y_test_prod, pred_test,
        labels=[cls], average='macro',
        zero_division=0)
    f = f1_score(y_test_prod, pred_test,
        labels=[cls], average='macro',
        zero_division=0)
    print(f"{cls:<20} {p:>10.3f} {r:>8.3f} {f:>8.3f} {sup:>10}")

print("\nWeakest class: Credit Card F1=0.68")
print("Strongest class: Credit Reporting F1=0.93")

## Key Findings

- **TF-IDF + LR = Champion** among classical models
- Word2Vec worse — averaging loses domain keywords!
- `credit report`, `debt collector`, `mortgage` are strong TF-IDF features

**Next → 03_bert_embeddings.ipynb**